# Meltwater Hangman — CANINE-S Style Character Transformer

Self-contained notebook: trains a CANINE-S style character-level Transformer from scratch on synthetic
Monte-Carlo Hangman states (Phase 1), then fine-tunes via self-play (Phase 2), and generates a
Kaggle submission CSV. No pretrained weights, no internet, no `transformers` library.

**Architecture**: 3.19M-param TransformerEncoder (hidden=256, layers=4, heads=4) with CLS pooling → 26-letter
logit classifier. Fixed 60-token layout: `[CLS] guessed[0..25] [SEP] word[0..31]`.

**Training**: Monte-Carlo biased random sampler (40% correct-letter bias) with frequency-weighted soft
target cross-entropy, followed by greedy self-play fine-tuning with MC-mixed batches.

In [ ]:
# ─── Cell 0: Imports ─────────────────────────────────────────────────────────
import os, csv, math, time, random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

# ─── Cell 1: Configuration ───────────────────────────────────────────────────
class Cfg:
    SEED = 1920
    DATA_DIR = "../input/meltwater-hangman"
    TRAIN_FILE = os.path.join(DATA_DIR, "train.txt")
    TEST_FILE  = os.path.join(DATA_DIR, "test.txt")
    CHECKPOINT_DIR = "checkpoints"
    BEST_MODEL_PATH = os.path.join(CHECKPOINT_DIR, "best_model.pt")
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)

    # Alphabet & tokens
    ALPHABET = "abcdefghijklmnopqrstuvwxyz"
    A_CODE = ord("a")
    NUM_LETTERS = 26
    PAD = 26; MASK = 27; CLS = 28; SEP = 29; VOCAB_SIZE = 30

    # Model
    MODEL_DIM = 256; NUM_HEADS = 4; NUM_LAYERS = 4
    FF_DIM = 1024; DROPOUT = 0.1
    MAX_SEQ_LEN = 64; MAX_WORD_LEN = 32

    # Layout: CLS(1) + guessed(26) + SEP(1) + word(32) = 60 <= 64
    LAYOUT_GUESS_START = 1; LAYOUT_SEP = 27; LAYOUT_WORD_START = 28

    # Training
    BATCH_SIZE = 512; VAL_BATCH_SIZE = 4096
    PHASE1_EPOCHS = 3
    SELF_PLAY_ROUNDS = 2; SELF_PLAY_EPOCHS_PER_ROUND = 1
    MAX_TRAIN_STEPS = None; MAX_MC_WORDS_PER_EPOCH = 0
    SELF_PLAY_BATCH = 4096; MC_MIX_RATIO = 0.30
    MAX_WRONG_GUESSES = 6; CORRECT_GUESS_PROB = 0.40
    VAL_SPLIT_RATIO = 0.10; FINAL_FULL_DATA = False

    # Optimiser
    LR = 3e-4; WEIGHT_DECAY = 1e-4
    LR_SCHEDULER = "cosine"; WARMUP_STEPS = 500; MAX_GRAD_NORM = 1.0

    # Device
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    USE_AMP = torch.cuda.is_available()
    VAL_WORDS = None

cfg = Cfg()
LAYOUT_LEN = 1 + cfg.NUM_LETTERS + 1 + cfg.MAX_WORD_LEN  # 60
print(f"device={cfg.DEVICE}  amp={cfg.USE_AMP}")

In [ ]:
# ─── Cell 2: Helpers ─────────────────────────────────────────────────────────
ALPHABET = cfg.ALPHABET
CHAR_TO_ID = {ch: i for i, ch in enumerate(ALPHABET)}
ID_TO_CHAR = {i: ch for ch, i in CHAR_TO_ID.items()}

def set_seed(seed=cfg.SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def word_to_ids(word):
    return [CHAR_TO_ID[c] for c in word]

def load_words(path):
    words = []
    with open(path, "r", encoding="utf-8") as f:
        for line in tqdm(f, desc=f"Loading {os.path.basename(path)}"):
            w = line.strip().lower()
            if w: words.append(w)
    return words

def make_split(train_words):
    rng = np.random.RandomState(cfg.SEED)
    idx = rng.permutation(len(train_words))
    n_val = int(len(train_words) * cfg.VAL_SPLIT_RATIO)
    val_idx = idx[:n_val]
    train_idx = idx[n_val:]
    return [train_words[i] for i in train_idx], [train_words[i] for i in val_idx]

def soft_target_vector(word, guessed_mask):
    counts = np.zeros(cfg.NUM_LETTERS, dtype=np.float64)
    total = 0
    for c in word:
        ci = CHAR_TO_ID[c]
        if guessed_mask[ci]: continue
        counts[ci] += 1; total += 1
    out = np.zeros(cfg.NUM_LETTERS, dtype=np.float64)
    if total > 0: out = counts / total
    return out.astype(np.float32)

print("Helpers OK")

In [ ]:
# ─── Cell 3: Monte-Carlo Simulator ───────────────────────────────────────────
def simulate_game_mc(word, rng, correct_rate=cfg.CORRECT_GUESS_PROB,
                     max_wrong=cfg.MAX_WRONG_GUESSES):
    word_ids = np.array(word_to_ids(word), dtype=np.int64)
    n = len(word_ids)
    revealed = np.zeros(n, dtype=bool)
    guessed_mask = np.zeros(cfg.NUM_LETTERS, dtype=bool)
    guessed_chars = []
    num_wrong = 0
    states, targets = [], []
    while (not revealed.all()) and num_wrong < max_wrong:
        tgt = soft_target_vector(word, guessed_mask)
        states.append(("".join(guessed_chars), tuple(revealed.tolist()), word))
        targets.append(tgt)
        unguessed = [c for c in ALPHABET if not guessed_mask[CHAR_TO_ID[c]]]
        if not unguessed: break
        correct_pool = [c for c in unguessed if c in word]
        if correct_pool and rng.random() < correct_rate:
            guess = rng.choice(correct_pool)
        else:
            guess = rng.choice(unguessed)
        ci = CHAR_TO_ID[guess]
        guessed_mask[ci] = True; guessed_chars.append(guess)
        if guess in word:
            revealed[word_ids == ci] = True
        else:
            num_wrong += 1
    return states, np.array(targets), bool(revealed.all())

# Tokeniser: fixed layout [CLS] guessed[0..25] [SEP] word[0..MAX_WORD_LEN-1]
def encode_state(guessed_chars, revealed_flags, word, max_len=LAYOUT_LEN):
    ids = [cfg.CLS] + [cfg.PAD] * (max_len - 1)
    attn = [1] * max_len
    for i, c in enumerate(guessed_chars[:cfg.NUM_LETTERS]):
        ids[cfg.LAYOUT_GUESS_START + i] = CHAR_TO_ID[c]
    ids[cfg.LAYOUT_SEP] = cfg.SEP
    for i, c in enumerate(word[:cfg.MAX_WORD_LEN]):
        pos = cfg.LAYOUT_WORD_START + i
        if pos < max_len:
            ids[pos] = CHAR_TO_ID[c] if revealed_flags[i] else cfg.MASK
    return ids, attn

class HangmanStateDataset(Dataset):
    def __init__(self, input_ids, attn_mask, targets):
        self.input_ids = input_ids; self.attn_mask = attn_mask; self.targets = targets
    def __len__(self): return self.input_ids.shape[0]
    def __getitem__(self, idx): return (self.input_ids[idx], self.attn_mask[idx], self.targets[idx])

def generate_mc_states(words, rng=None, desc="MC states"):
    if rng is None: rng = np.random.RandomState(cfg.SEED)
    all_states, all_targets = [], []
    for w in tqdm(words, desc=desc):
        states, targets, _ = simulate_game_mc(w, rng)
        all_states.extend(states); all_targets.extend(targets)
    n = len(all_states)
    ids_arr = np.full((n, LAYOUT_LEN), cfg.PAD, dtype=np.int64)
    attn_arr = np.ones((n, LAYOUT_LEN), dtype=np.int64)
    for i, (gstr, rev, word) in enumerate(all_states):
        ids, am = encode_state(gstr, rev, word)
        ids_arr[i] = ids; attn_arr[i] = am
    return HangmanStateDataset(ids_arr, attn_arr, np.array(all_targets, dtype=np.float32))

def collate(batch):
    ids = torch.tensor(np.stack([b[0] for b in batch]), dtype=torch.long)
    am  = torch.tensor(np.stack([b[1] for b in batch]), dtype=torch.long)
    tgt = torch.tensor(np.stack([b[2] for b in batch]), dtype=torch.float32)
    return ids, am, tgt

print("Simulator OK")

In [ ]:
# ─── Cell 4: Model ────────────────────────────────────────────────────────────
class PositionalEncoding(nn.Module):
    def __init__(self, dim, max_len):
        super().__init__()
        pe = torch.zeros(max_len, dim)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, dim, 2).float() * (-math.log(10000.0) / dim))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x): return x + self.pe[:, :x.size(1)]

class CanineHangmanModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_emb = nn.Embedding(cfg.VOCAB_SIZE, cfg.MODEL_DIM, padding_idx=cfg.PAD)
        self.pos_enc = PositionalEncoding(cfg.MODEL_DIM, cfg.MAX_SEQ_LEN)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=cfg.MODEL_DIM, nhead=cfg.NUM_HEADS, dim_feedforward=cfg.FF_DIM,
            dropout=cfg.DROPOUT, activation="gelu", batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=cfg.NUM_LAYERS,
                                            enable_nested_tensor=False)
        self.dropout = nn.Dropout(cfg.DROPOUT)
        self.classifier = nn.Linear(cfg.MODEL_DIM, cfg.NUM_LETTERS)

    def forward(self, input_ids, attn_mask=None):
        x = self.token_emb(input_ids)
        x = self.pos_enc(x)
        x = self.dropout(x)
        key_padding_mask = (attn_mask == 0) if attn_mask is not None else None
        x = self.encoder(x, src_key_padding_mask=key_padding_mask)
        return self.classifier(self.dropout(x[:, 0]))

class HangmanAgent:
    def __init__(self, model=None):
        self.model = model or CanineHangmanModel()
        self.device = next(self.model.parameters()).device
    def to(self, d): self.model = self.model.to(d); self.device = d; return self
    def eval(self): self.model.eval()
    def train(self): self.model.train()
    @torch.no_grad()
    def predict(self, input_ids, attn_mask, guessed_mask=None):
        logits = self.model(input_ids.to(self.device), attn_mask.to(self.device))
        if guessed_mask is not None:
            logits = logits.masked_fill(guessed_mask.to(self.device), float("-inf"))
        nxt = logits.argmax(dim=-1).cpu()
        if guessed_mask is not None:
            all_g = guessed_mask.all(dim=-1)
            nxt = torch.where(all_g.cpu(), torch.zeros_like(nxt), nxt)
        return nxt

m = CanineHangmanModel()
print(f"Model params: {sum(p.numel() for p in m.parameters()):,}")
del m

In [ ]:
# ─── Cell 5: Self-play + Validation ──────────────────────────────────────────
def _build_chunk_tokens(chunk_words, revealed, guessed_char_lists, max_len):
    B = len(chunk_words)
    ids = np.full((B, max_len), cfg.PAD, dtype=np.int64)
    attn = np.ones((B, max_len), dtype=np.int64)
    gm = np.zeros((B, cfg.NUM_LETTERS), dtype=bool)
    ids[:, 0] = cfg.CLS; ids[:, cfg.LAYOUT_SEP] = cfg.SEP
    for b, gstr in enumerate(guessed_char_lists):
        for i, c in enumerate(gstr[:cfg.NUM_LETTERS]):
            ids[b, cfg.LAYOUT_GUESS_START + i] = CHAR_TO_ID[c]
            gm[b, CHAR_TO_ID[c]] = True
    for b, (word, rev) in enumerate(zip(chunk_words, revealed)):
        for i, c in enumerate(word[:cfg.MAX_WORD_LEN]):
            pos = cfg.LAYOUT_WORD_START + i
            if pos < max_len:
                ids[b, pos] = CHAR_TO_ID[c] if rev[i] else cfg.MASK
    return torch.from_numpy(ids), torch.from_numpy(attn), torch.from_numpy(gm)

def _guessed_bool(gstr):
    m = np.zeros(cfg.NUM_LETTERS, dtype=bool)
    for c in gstr: m[CHAR_TO_ID[c]] = True
    return m

def rollout_selfplay(agent, words, batch_size=cfg.SELF_PLAY_BATCH, desc="Self-play"):
    agent.eval()
    device = agent.device
    amp_ctx = torch.amp.autocast("cuda", enabled=(cfg.USE_AMP and device.type != "cpu"))
    all_ids, all_attn, all_tgt = [], [], []
    for start in tqdm(range(0, len(words), batch_size), desc=desc):
        chunk = words[start:start+batch_size]; B = len(chunk)
        W = min(max(len(w) for w in chunk), cfg.MAX_WORD_LEN)
        wids = [[CHAR_TO_ID[c] for c in w] for w in chunk]
        wlens = np.array([len(w) for w in chunk], dtype=np.int64)
        wids_arr = np.full((B, W), -1, dtype=np.int64)
        for b, wi in enumerate(wids): wids_arr[b, :min(len(wi), W)] = wi[:min(len(wi), W)]
        revealed = np.zeros((B, W), dtype=bool)
        gcl = [""] * B; wrong = np.zeros(B, dtype=np.int64); done = np.zeros(B, dtype=bool)
        step_states = []
        for _ in range(cfg.NUM_LETTERS):
            if done.all(): break
            aidx = np.where(~done)[0]
            if len(aidx) == 0: break
            a_words = [chunk[i] for i in aidx]
            ids, am, gm = _build_chunk_tokens(a_words, revealed[aidx],
                                             [gcl[i] for i in aidx], LAYOUT_LEN)
            with amp_ctx: logits = agent.model(ids.to(device), am.to(device))
            logits = logits.masked_fill(gm.to(device), float("-inf"))
            nxt = logits.argmax(dim=-1).cpu().numpy()
            for j, bi in enumerate(aidx):
                tgt = soft_target_vector(chunk[bi], _guessed_bool(gcl[bi]))
                step_states.append((ids[j].numpy(), am[j].numpy(), tgt))
            for j, bi in enumerate(aidx):
                gid = int(nxt[j])
                if chr(cfg.A_CODE + gid) in gcl[bi]:
                    for cid in range(26):
                        if chr(cfg.A_CODE + cid) not in gcl[bi]: gid = cid; break
                gc = chr(cfg.A_CODE + gid); gcl[bi] += gc
                if gc in chunk[bi]:
                    revealed[bi] |= (wids_arr[bi] == gid)
                    if min(wlens[bi], W) > 0 and revealed[bi, :min(wlens[bi], W)].all(): done[bi] = True
                else:
                    wrong[bi] += 1
                    if wrong[bi] >= cfg.MAX_WRONG_GUESSES: done[bi] = True
        for ids_j, am_j, tgt_j in step_states:
            all_ids.append(ids_j); all_attn.append(am_j); all_tgt.append(tgt_j)
    if not all_ids:
        return (torch.zeros(0, LAYOUT_LEN, dtype=torch.long),
                torch.zeros(0, LAYOUT_LEN, dtype=torch.long),
                torch.zeros(0, cfg.NUM_LETTERS, dtype=torch.float32))
    return (torch.from_numpy(np.stack(all_ids)),
            torch.from_numpy(np.stack(all_attn)),
            torch.from_numpy(np.stack(all_tgt)))

def validate(agent, val_words, batch_size=cfg.VAL_BATCH_SIZE, desc="Validation"):
    agent.eval(); device = agent.device
    amp_ctx = torch.amp.autocast("cuda", enabled=(cfg.USE_AMP and device.type != "cpu"))
    total_solved = 0; total_wrong = 0; total_guesses = 0; total = len(val_words)
    for start in tqdm(range(0, total, batch_size), desc=desc):
        chunk = val_words[start:start+batch_size]; B = len(chunk)
        W = min(max(len(w) for w in chunk), cfg.MAX_WORD_LEN)
        wids = [[CHAR_TO_ID[c] for c in w] for w in chunk]
        wlens = np.array([len(w) for w in chunk], dtype=np.int64)
        wids_arr = np.full((B, W), -1, dtype=np.int64)
        for b, wi in enumerate(wids): wids_arr[b, :min(len(wi), W)] = wi[:min(len(wi), W)]
        revealed = np.zeros((B, W), dtype=bool)
        gcl = [""] * B; wrong = np.zeros(B, dtype=np.int64)
        tot_g = np.zeros(B, dtype=np.int64); solved = np.zeros(B, dtype=bool)
        done = np.zeros(B, dtype=bool)
        for _ in range(cfg.NUM_LETTERS):
            if done.all(): break
            aidx = np.where(~done)[0]
            if len(aidx) == 0: break
            a_words = [chunk[i] for i in aidx]
            ids, am, gm = _build_chunk_tokens(a_words, revealed[aidx],
                                             [gcl[i] for i in aidx], LAYOUT_LEN)
            with amp_ctx: logits = agent.model(ids.to(device), am.to(device))
            logits = logits.masked_fill(gm.to(device), float("-inf"))
            nxt = logits.argmax(dim=-1).cpu().numpy()
            for j, bi in enumerate(aidx):
                gid = int(nxt[j])
                if chr(cfg.A_CODE + gid) in gcl[bi]:
                    for cid in range(26):
                        if chr(cfg.A_CODE + cid) not in gcl[bi]: gid = cid; break
                gc = chr(cfg.A_CODE + gid); gcl[bi] += gc; tot_g[bi] += 1
                if gc in chunk[bi]:
                    revealed[bi] |= (wids_arr[bi] == gid)
                    if min(wlens[bi], W) > 0 and revealed[bi, :min(wlens[bi], W)].all():
                        solved[bi] = True; done[bi] = True
                else:
                    wrong[bi] += 1
                    if wrong[bi] >= cfg.MAX_WRONG_GUESSES: done[bi] = True
        total_solved += int(solved.sum()); total_wrong += int(wrong.sum()); total_guesses += int(tot_g.sum())
    wr = total_solved / max(1, total)
    print(f"[Val] win_rate={wr*100:.2f}% solved={total_solved}/{total} avg_wrong={total_wrong/max(1,total):.3f}")
    return {"win_rate": wr, "avg_wrong": total_wrong/max(1,total), "solved": total_solved, "total": total}

print("Self-play + validation OK")

In [ ]:
# ─── Cell 6: Training ────────────────────────────────────────────────────────
def make_optim(model):
    decay, no_decay = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad: continue
        (no_decay if p.ndim < 2 else decay).append(p)
    return torch.optim.AdamW(
        [{"params": decay, "weight_decay": cfg.WEIGHT_DECAY},
         {"params": no_decay, "weight_decay": 0.0}], lr=cfg.LR)

def make_scheduler(opt, total_steps):
    if cfg.LR_SCHEDULER == "cosine":
        return torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, total_steps))
    elif cfg.LR_SCHEDULER == "linear_warmup":
        def lr_lambda(step):
            if step < cfg.WARMUP_STEPS: return step / max(1, cfg.WARMUP_STEPS)
            return max(0.0, (total_steps - step) / max(1, total_steps - cfg.WARMUP_STEPS))
        return torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
    return torch.optim.lr_scheduler.StepLR(opt, step_size=max(1, total_steps), gamma=1.0)

def train_epoch(model, dataset, optim, scheduler, scaler, loss_fn, epoch, phase,
                max_steps=None, device=cfg.DEVICE):
    model.train()
    loader = DataLoader(dataset, batch_size=cfg.BATCH_SIZE, shuffle=True,
                        collate_fn=collate, drop_last=False)
    if len(dataset) == 0: return 0.0, 0
    use_amp = cfg.USE_AMP and device.type != "cpu"
    running_loss = 0.0; seen = 0; step = 0
    pbar = tqdm(loader, desc=f"[{phase} e{epoch}]", unit="batch")
    for ids, am, tgt in pbar:
        ids, am, tgt = ids.to(device), am.to(device), tgt.to(device)
        optim.zero_grad()
        with torch.amp.autocast("cuda", enabled=use_amp):
            logits = model(ids, am)
        loss = loss_fn(logits, tgt)
        if use_amp:
            scaler.scale(loss).backward()
            scaler.unscale_(optim)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.MAX_GRAD_NORM)
            scaler.step(optim); scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.MAX_GRAD_NORM)
            optim.step()
        if scheduler: scheduler.step()
        running_loss += loss.item(); seen += 1; step += 1
        pbar.set_postfix({"loss": f"{running_loss/max(1,seen):.4f}"})
        if max_steps and step >= max_steps: break
    return running_loss / max(1, seen), step

class MixedDataset(Dataset):
    def __init__(self, sp_ids, sp_attn, sp_tgt, mc_words, seed):
        self.sp_ids, self.sp_attn, self.sp_tgt = sp_ids, sp_attn, sp_tgt
        mc_ds = generate_mc_states(mc_words[:max(1, len(mc_words)//2)],
                                   np.random.RandomState(seed + 7), desc="MC mix")
        n_mc = max(1, int(len(sp_ids) * cfg.MC_MIX_RATIO))
        idx = np.random.RandomState(seed + 3).permutation(len(mc_ds))[:n_mc]
        self.mc_ids = torch.from_numpy(mc_ds.input_ids[idx])
        self.mc_attn = torch.from_numpy(mc_ds.attn_mask[idx])
        self.mc_tgt = torch.from_numpy(mc_ds.targets[idx])
    def __len__(self): return len(self.sp_ids)
    def __getitem__(self, idx):
        if idx < len(self.sp_ids):
            return (self.sp_ids[idx], self.sp_attn[idx], self.sp_tgt[idx])
        m = idx - len(self.sp_ids)
        return (self.mc_ids[m], self.mc_attn[m], self.mc_tgt[m])

print("Training utils OK")

In [ ]:
# ─── Cell 7: Load data + run pipeline ────────────────────────────────────────
set_seed()
train_words_full, test_words = load_words(cfg.TRAIN_FILE), load_words(cfg.TEST_FILE)
train_words, val_words = make_split(train_words_full)
print(f"train={len(train_words)} val={len(val_words)} test={len(test_words)}")

device = torch.device(cfg.DEVICE)
model = CanineHangmanModel().to(device)
agent = HangmanAgent(model)
loss_fn = nn.CrossEntropyLoss()
scaler = torch.amp.GradScaler("cuda", enabled=(cfg.USE_AMP and device.type != "cpu"))

mc_words = train_words[:cfg.MAX_MC_WORDS_PER_EPOCH] if cfg.MAX_MC_WORDS_PER_EPOCH else train_words
best_val = -1.0

# ── Phase 1: Monte-Carlo supervised ──
for epoch in range(1, cfg.PHASE1_EPOCHS + 1):
    print(f"\n=== Phase 1 | epoch {epoch} ===")
    dataset = generate_mc_states(mc_words, np.random.RandomState(cfg.SEED + epoch),
                                 desc=f"MC e{epoch}")
    opt = make_optim(model)
    sched = make_scheduler(opt, max(1, len(dataset) // cfg.BATCH_SIZE))
    scaler = torch.amp.GradScaler("cuda", enabled=(cfg.USE_AMP and device.type != "cpu"))
    loss, steps = train_epoch(model, dataset, opt, sched, scaler, loss_fn, epoch, "Phase1",
                              max_steps=cfg.MAX_TRAIN_STEPS, device=device)
    agent.model = model
    metrics = validate(agent, val_words, desc=f"Val Phase1 e{epoch}")
    if metrics["win_rate"] > best_val:
        best_val = metrics["win_rate"]
        torch.save({"model_state_dict": model.state_dict(), "metrics": metrics},
                   cfg.BEST_MODEL_PATH)
        print(f"[best] {best_val:.4f}")

# ── Phase 2: Self-play ──
if cfg.SELF_PLAY_ROUNDS > 0 and os.path.exists(cfg.BEST_MODEL_PATH):
    ckpt = torch.load(cfg.BEST_MODEL_PATH, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    agent = HangmanAgent(model)

for rnd in range(1, cfg.SELF_PLAY_ROUNDS + 1):
    for sub in range(cfg.SELF_PLAY_EPOCHS_PER_ROUND):
        print(f"\n=== Phase 2 | round {rnd} epoch {sub+1} ===")
        sp_ids, sp_attn, sp_tgt = rollout_selfplay(agent, mc_words,
                                                   desc=f"SP r{rnd}e{sub+1}")
        dataset = MixedDataset(sp_ids, sp_attn, sp_tgt, mc_words,
                               cfg.SEED + 1000 + rnd * 10 + sub)
        opt = make_optim(model)
        sched = make_scheduler(opt, max(1, len(dataset) // cfg.BATCH_SIZE))
        scaler = torch.amp.GradScaler("cuda", enabled=(cfg.USE_AMP and device.type != "cpu"))
        loss, steps = train_epoch(model, dataset, opt, sched, scaler, loss_fn, sub+1,
                                  f"SP r{rnd}", max_steps=cfg.MAX_TRAIN_STEPS, device=device)
        agent.model = model
        metrics = validate(agent, val_words, desc=f"Val SP r{rnd}e{sub+1}")
        if metrics["win_rate"] > best_val:
            best_val = metrics["win_rate"]
            torch.save({"model_state_dict": model.state_dict(), "metrics": metrics},
                       cfg.BEST_MODEL_PATH)
            print(f"[best] {best_val:.4f}")

final_path = os.path.join(cfg.CHECKPOINT_DIR, "final_model.pt")
torch.save({"model_state_dict": model.state_dict(), "best_val": best_val}, final_path)
print(f"\n[done] best_val={best_val:.4f} model={final_path}")

In [ ]:
# ─── Cell 8: Generate Kaggle submission ──────────────────────────────────────
def generate_submission(agent, test_words, batch_size=cfg.VAL_BATCH_SIZE, out_csv="submission.csv"):
    agent.eval(); device = agent.device
    amp_ctx = torch.amp.autocast("cuda", enabled=(cfg.USE_AMP and device.type != "cpu"))
    rows = []
    for start in tqdm(range(0, len(test_words), batch_size), desc="Submission"):
        chunk = test_words[start:start+batch_size]; B = len(chunk)
        W = min(max(len(w) for w in chunk), cfg.MAX_WORD_LEN)
        wids = [[CHAR_TO_ID[c] for c in w] for w in chunk]
        wlens = np.array([len(w) for w in chunk], dtype=np.int64)
        wids_arr = np.full((B, W), -1, dtype=np.int64)
        for b, wi in enumerate(wids): wids_arr[b, :min(len(wi), W)] = wi[:min(len(wi), W)]
        revealed = np.zeros((B, W), dtype=bool)
        gcl = [""] * B; wrong = np.zeros(B, dtype=np.int64); done = np.zeros(B, dtype=bool)
        for _ in range(cfg.NUM_LETTERS):
            if done.all(): break
            aidx = np.where(~done)[0]
            if len(aidx) == 0: break
            a_words = [chunk[i] for i in aidx]
            ids, am, gm = _build_chunk_tokens(a_words, revealed[aidx],
                                             [gcl[i] for i in aidx], LAYOUT_LEN)
            with amp_ctx: logits = agent.model(ids.to(device), am.to(device))
            logits = logits.masked_fill(gm.to(device), float("-inf"))
            nxt = logits.argmax(dim=-1).cpu().numpy()
            for j, bi in enumerate(aidx):
                gid = int(nxt[j])
                if chr(cfg.A_CODE + gid) in gcl[bi]:
                    for cid in range(26):
                        if chr(cfg.A_CODE + cid) not in gcl[bi]: gid = cid; break
                gc = chr(cfg.A_CODE + gid); gcl[bi] += gc
                if gc in chunk[bi]:
                    revealed[bi] |= (wids_arr[bi] == gid)
                    if min(wlens[bi], W) > 0 and revealed[bi, :min(wlens[bi], W)].all(): done[bi] = True
                else:
                    wrong[bi] += 1
                    if wrong[bi] >= cfg.MAX_WRONG_GUESSES: done[bi] = True
        for bi in range(B): rows.append((start + bi, gcl[bi]))
    rows.sort(key=lambda r: r[0])
    with open(out_csv, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f); w.writerow(["word_id", "guessed_letters_string"])
        for wid, gstr in rows: w.writerow([wid, gstr])
    print(f"Wrote {len(rows)} rows -> {out_csv}")
    return out_csv

# Load best model
if os.path.exists(cfg.BEST_MODEL_PATH):
    ckpt = torch.load(cfg.BEST_MODEL_PATH, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    print(f"Loaded best model: {ckpt.get('metrics', {}).get('win_rate', 0):.4f}")
else:
    print("No checkpoint found, using current model.")

generate_submission(agent, test_words)
print("Done!")